In [5]:
# ============================================================
# 🚀 PIPELINE FINAL: METABOLISMO + CLÍNICO + KM (VERSIÓN 2026)
# ============================================================

# 1. CARGA DE LIBRERÍAS
required_libs <- c("dplyr", "ggplot2", "reshape2", "ggpubr", "survival",
                   "survminer", "readxl", "purrr", "tidyr", "grid", "gridExtra",
                   "scales")
invisible(lapply(required_libs, library, character.only = TRUE))

# ===============================
# 2. CONFIGURACIÓN DE RUTAS
# ===============================
paths <- list(
  metabolic    = "/Users/eduardoruiz/Documents/GitHub/Precision-Oncology-for-Breast-Cancer-Diagnosis/src/Clustering and data analysis PYTHON/Metabolic data analysis/ML models using metabolic data/resultados_TumorPhenotype_PCA_metrics/Merged_TumorPhenotype_PCA_AllData.csv",
  clinicaldata = "/Users/eduardoruiz/Documents/GitHub/Precision-Oncology-for-Breast-Cancer-Diagnosis/src/Clustering and data analysis PYTHON/Clinical data analysis/ML models using clinical data/Results_clustering_UMAP/DATA_MASTER_CLUSTERS_Y_CLINICA.csv",
  div          = "/Users/eduardoruiz/Documents/GitHub/Precision-Oncology-for-Breast-Cancer-Diagnosis/src/Clustering and data analysis PYTHON/Cluster_correlations/pacientes_divergentes_para_estudio_pyn.csv",
  core         = "/Users/eduardoruiz/Documents/GitHub/Precision-Oncology-for-Breast-Cancer-Diagnosis/src/Clustering and data analysis PYTHON/Cluster_correlations/pacientes_core_correlacion_Paretoynormas.csv"
)

group_colors <- c("Core" = "#0072B2", "Divergent" = "#D55E00")

# ─── TEMA GLOBAL ────────────────────────────────────────────────────────────
base_theme <- theme_pubr() +
  theme(
    title         = element_text(size = 18, face = "bold"),
    axis.title    = element_text(size = 15),
    axis.text     = element_text(size = 13),
    legend.text   = element_text(size = 13),
    legend.title  = element_text(size = 14, face = "bold"),
    strip.text    = element_text(size = 14, face = "bold"),
    plot.subtitle = element_text(size = 13, color = "grey50")
  )

# ===============================
# 3. VARIABLES METABÓLICAS
# ===============================

# ─── Lista completa de variables metabólicas nuevas ─────────────────────────
ALL_METAB_VARS <- c(
  # Capacidad de Uso / Eficiencia de Activación
  "CU_FBA", "CU_pFBA", "CU_L1w",
  "EA_FBA", "EA_pFBA", "EA_L1w",
  # Índice de Warburg
  "WarburgIndex_FBA", "WarburgIndex_pFBA", "WarburgIndex_L1w",
  # Síntesis de ácidos grasos
  "SA_Fatty_acid_synthesis_FBA", "SA_Fatty_acid_synthesis_pFBA", "SA_Fatty_acid_synthesis_L1w",
  # Metabolismo de glutamato
  "SA_Glutamate_metabolism_FBA", "SA_Glutamate_metabolism_pFBA", "SA_Glutamate_metabolism_L1w",
  # Fosforilación oxidativa
  "SA_Oxidative_phosphorylation_FBA", "SA_Oxidative_phosphorylation_pFBA", "SA_Oxidative_phosphorylation_L1w",
  # Índice Redox
  "RedoxIndex_FBA", "RedoxIndex_pFBA", "RedoxIndex_L1w",
  # Índice de Flujo Metabólico
  "MFI_FBA", "MFI_pFBA", "MFI_L1w",
  # Score de Anabolismo (sin pFBA)
  "AnabolismScore_FBA", "AnabolismScore_L1w",
  # Demanda de NADPH (sin pFBA)
  "NADPHdemand_FBA", "NADPHdemand_L1w",
  # Lípidos saturados y fosfolípidos (sin pFBA)
  "LipidSat_FBA", "LipidSat_L1w",
  "LipidPL_FBA",  "LipidPL_L1w",
  # Dependencia de glutamina (sin pFBA)
  "GlnDependence_FBA", "GlnDependence_L1w",
  # Oncometabolitos (sin pFBA)
  "Oncomet_Lactate_FBA",   "Oncomet_Lactate_L1w",
  "Oncomet_Succinate_FBA", "Oncomet_Succinate_L1w",
  "Oncomet_Fumarate_FBA",  "Oncomet_Fumarate_L1w"
)

# Grupos temáticos para seccionar el reporte (usados en títulos de página)
METAB_GROUPS <- list(
  "Capacidad de Uso y Eficiencia de Activación" = c(
    "CU_FBA", "CU_pFBA", "CU_L1w",
    "EA_FBA", "EA_pFBA", "EA_L1w"
  ),
  "Índice de Warburg" = c(
    "WarburgIndex_FBA", "WarburgIndex_pFBA", "WarburgIndex_L1w"
  ),
  "Vías Metabólicas (SA)" = c(
    "SA_Fatty_acid_synthesis_FBA", "SA_Fatty_acid_synthesis_pFBA", "SA_Fatty_acid_synthesis_L1w",
    "SA_Glutamate_metabolism_FBA", "SA_Glutamate_metabolism_pFBA", "SA_Glutamate_metabolism_L1w",
    "SA_Oxidative_phosphorylation_FBA", "SA_Oxidative_phosphorylation_pFBA", "SA_Oxidative_phosphorylation_L1w"
  ),
  "Redox, Flujo y Anabolismo" = c(
    "RedoxIndex_FBA", "RedoxIndex_pFBA", "RedoxIndex_L1w",
    "MFI_FBA", "MFI_pFBA", "MFI_L1w",
    "AnabolismScore_FBA", "AnabolismScore_L1w",
    "NADPHdemand_FBA", "NADPHdemand_L1w"
  ),
  "Lípidos y Glutamina" = c(
    "LipidSat_FBA", "LipidSat_L1w",
    "LipidPL_FBA",  "LipidPL_L1w",
    "GlnDependence_FBA", "GlnDependence_L1w"
  ),
  "Oncometabolitos" = c(
    "Oncomet_Lactate_FBA",   "Oncomet_Lactate_L1w",
    "Oncomet_Succinate_FBA", "Oncomet_Succinate_L1w",
    "Oncomet_Fumarate_FBA",  "Oncomet_Fumarate_L1w"
  )
)

# ===============================
# 4. PREPROCESAMIENTO Y CRUCE
# ===============================
clean_names <- function(x) toupper(gsub("\\.", "-", trimws(as.character(x))))

df_metab <- read.csv(paths$metabolic,    stringsAsFactors = FALSE)
df_clin  <- read.csv(paths$clinicaldata, stringsAsFactors = FALSE)
df_div   <- read.csv(paths$div,          stringsAsFactors = FALSE)
df_core  <- read.csv(paths$core,         stringsAsFactors = FALSE)

df_metab$ModelName <- clean_names(df_metab$ModelName)
df_clin$ModelName  <- clean_names(df_clin$ModelName)
div_names          <- clean_names(df_div$ModelName)
core_names         <- clean_names(df_core$ModelName)

# ─── FIX 1: Detectar colisiones entre grupos ────────────────────────────────
overlap <- intersect(div_names, core_names)
if (length(overlap) > 0) {
  warning(paste0("⚠️  ", length(overlap), " pacientes en ambos grupos (se excluirán): ",
                 paste(head(overlap, 5), collapse = ", ")))
}

analysis <- df_metab %>%
  mutate(Group = case_when(
    ModelName %in% overlap    ~ NA_character_,
    ModelName %in% div_names  ~ "Divergent",
    ModelName %in% core_names ~ "Core",
    TRUE                      ~ "Other"
  )) %>%
  filter(Group %in% c("Core", "Divergent"))

# ─── Verificar qué variables metabólicas están realmente en el CSV ───────────
metab_available <- intersect(ALL_METAB_VARS, colnames(analysis))
metab_missing   <- setdiff(ALL_METAB_VARS, colnames(analysis))

cat("\n📊 Variables metabólicas encontradas:", length(metab_available), "/", length(ALL_METAB_VARS))
if (length(metab_missing) > 0) {
  cat("\n⚠️  Variables NO encontradas en el CSV:\n")
  cat(paste0("   • ", metab_missing, collapse = "\n"), "\n")
}

# ─── FIX 2: Join clínico ────────────────────────────────────────────────────
clinical_cols <- c(
  "ModelName",
  "ajcc_pathologic_stage.diagnoses",
  "ajcc_pathologic_t.diagnoses",
  "ajcc_pathologic_n.diagnoses",
  "ajcc_pathologic_m.diagnoses",
  "tumor_grade.diagnoses",
  "morphology.diagnoses",
  "primary_diagnosis.diagnoses",
  "treatment_type.treatments.diagnoses",
  "treatment_or_therapy.treatments.diagnoses",
  "prior_treatment.diagnoses",
  "sample_type.samples",
  "tissue_type.samples",
  "tumor_descriptor.samples",
  "specimen_type.samples",
  "is_ffpe.samples",
  "oct_embedded.samples",
  "age_at_diagnosis.diagnoses",
  "alcohol_history.exposures",
  "Menopausal.Status",
  "Cancer.Type",
  "ER", "PR", "HER2",
  "Subtype",
  "Genetic.Ancestry",
  "Prior_Treatment_Flag",
  "Metastasis_Flag",
  "ER_PR_HER2_Combo",
  "ER_PR_HER2_Combo_encoded",
  "Subtype_encoded"
)

clin_subset <- df_clin %>% select(any_of(clinical_cols))

cols_from_clin <- setdiff(colnames(clin_subset), "ModelName")
analysis <- analysis %>%
  select(-any_of(cols_from_clin)) %>%
  left_join(clin_subset, by = "ModelName") %>%
  rename_with(~ gsub("Menopausal\\.Status", "Menopausal Status", .x)) %>%
  rename_with(~ gsub("Cancer\\.Type",       "Cancer Type",       .x)) %>%
  rename_with(~ gsub("Genetic\\.Ancestry",  "Genetic Ancestry",  .x))

cat("\n✅ Pacientes Core:", sum(analysis$Group == "Core"),
    " | Divergent:", sum(analysis$Group == "Divergent"), "\n")

# ===============================
# 5. ANÁLISIS ESTADÍSTICO (FDR)
# ===============================
run_stats <- function(data, columns) {
  results <- map_df(columns, function(col) {
    sub_data <- data %>%
      select(Group, value = !!sym(col)) %>%
      filter(is.finite(value), !is.na(value))

    group_sizes <- table(sub_data$Group)
    if (length(group_sizes) < 2 || any(group_sizes < 3)) return(NULL)

    test <- wilcox.test(value ~ Group, data = sub_data, exact = FALSE)

    meds <- sub_data %>%
      group_by(Group) %>%
      summarise(m = median(value), .groups = "drop")

    data.frame(
      Feature  = col,
      Med_Core = meds$m[meds$Group == "Core"],
      Med_Div  = meds$m[meds$Group == "Divergent"],
      p_val    = test$p.value
    )
  })

  if (nrow(results) == 0) return(results)

  results %>%
    mutate(
      p_adj      = p.adjust(p_val, method = "fdr"),
      Fold_Change = Med_Div / ifelse(Med_Core == 0, NA, Med_Core)
    ) %>%
    arrange(p_val)
}

# ─── Estadísticas para TODAS las variables metabólicas ──────────────────────
metab_res <- run_stats(analysis, metab_available)

cat("\n📈 Variables metabólicas significativas (p_adj < 0.05):",
    sum(metab_res$p_adj < 0.05, na.rm = TRUE), "/", nrow(metab_res), "\n")

# Exportar tabla de resultados estadísticos
write.csv(metab_res, "Estadisticas_Metabolicas_CoreVsDivergent.csv", row.names = FALSE)
cat("✅ Tabla estadística exportada: Estadisticas_Metabolicas_CoreVsDivergent.csv\n")

# Variables numéricas clínicas
numeric_clinical_cols <- intersect(
  c("age_at_diagnosis.diagnoses", "is_ffpe.samples", "oct_embedded.samples",
    "Prior_Treatment_Flag", "Metastasis_Flag", "ER_PR_HER2_Combo_encoded", "Subtype_encoded"),
  colnames(analysis)
)
clinical_num_res <- run_stats(analysis, numeric_clinical_cols)

# ===============================
# 6. FUNCIONES DE PLOTEO
# ===============================

# --- Violin + Boxplot con color de significancia ---
plot_box_feat <- function(feat, data, stats_df) {
  if (!feat %in% colnames(data)) return(NULL)
  p_info <- stats_df %>% filter(Feature == feat)
  if (nrow(p_info) == 0) return(NULL)
  p_info <- p_info %>% slice(1)

  plot_data <- data %>% filter(is.finite(!!sym(feat)), !is.na(!!sym(feat)))
  if (nrow(plot_data) == 0) return(NULL)

  sig_color <- if (!is.na(p_info$p_adj) && p_info$p_adj < 0.05) "#e63946" else "grey60"
  subtitle_txt <- paste0(
    "p-adj: ", signif(p_info$p_adj, 3),
    if (!is.na(p_info$Fold_Change)) paste0("  |  FC: ", signif(p_info$Fold_Change, 2)) else ""
  )

  ggplot(plot_data, aes(x = Group, y = !!sym(feat), fill = Group)) +
    geom_violin(alpha = 0.2, color = NA) +
    geom_boxplot(width = 0.4, outlier.shape = 16, outlier.size = 2) +
    labs(title = feat, subtitle = subtitle_txt, y = NULL) +
    scale_fill_manual(values = group_colors) +
    base_theme +
    theme(
      legend.position = "none",
      plot.title      = element_text(size = 12, color = sig_color, face = "bold"),
      plot.subtitle   = element_text(size = 10)
    )
}

# --- Heatmap de medianas por grupo (vista resumen) ---
plot_heatmap_summary <- function(data, stats_df, group_name = "Todas") {
  if (nrow(stats_df) == 0) return(NULL)

  hm_data <- stats_df %>%
    select(Feature, Med_Core, Med_Div) %>%
    pivot_longer(cols = c(Med_Core, Med_Div), names_to = "Group", values_to = "Median") %>%
    mutate(Group = recode(Group, "Med_Core" = "Core", "Med_Div" = "Divergent")) %>%
    group_by(Feature) %>%
    mutate(Median_scaled = scale(Median)[,1]) %>%
    ungroup()

  ggplot(hm_data, aes(x = Group, y = Feature, fill = Median_scaled)) +
    geom_tile(color = "white") +
    scale_fill_gradient2(low = "#0072B2", mid = "white", high = "#D55E00",
                         midpoint = 0, name = "Mediana\n(z-score)") +
    labs(title = paste("Heatmap:", group_name), x = "", y = "") +
    base_theme +
    theme(axis.text.y = element_text(size = 9))
}

# --- Variables clínicas: boxplot / barras ---
plot_clinical <- function(var, data) {
  if (!var %in% colnames(data)) return(NULL)
  tmp <- data %>% filter(!is.na(!!sym(var)))
  if (nrow(tmp) == 0) return(NULL)
  tmp[[var]] <- trimws(as.character(tmp[[var]]))

  if (is.numeric(tmp[[var]])) {
    ggplot(tmp, aes(x = Group, y = !!sym(var), fill = Group)) +
      geom_boxplot(outlier.shape = 16, outlier.size = 2) +
      stat_compare_means(label = "p.signif", label.size = 5) +
      scale_fill_manual(values = group_colors) +
      labs(title = var, x = "") +
      base_theme
  } else {
    tbl   <- table(tmp[[var]], tmp$Group)
    p_chi <- tryCatch(chisq.test(tbl)$p.value, error = function(e) NA)
    subtitle_txt <- if (!is.na(p_chi)) paste("Chi² p:", signif(p_chi, 3)) else ""

    ggplot(tmp, aes(x = !!sym(var), fill = Group)) +
      geom_bar(position = "fill") +
      scale_y_continuous(labels = percent) +
      scale_fill_manual(values = group_colors) +
      labs(title = var, subtitle = subtitle_txt, y = "Proporción") +
      base_theme +
      theme(axis.text.x = element_text(angle = 40, hjust = 1, size = 10))
  }
}

# ===============================
# 7. GENERACIÓN DEL REPORTE PDF
# ===============================
pdf("Reporte_Metabolismo_Clinico_2026.pdf", width = 14, height = 11)

# ─── Página de título ────────────────────────────────────────────────────────
grid.newpage()
grid.text("Reporte: Core vs Divergent\nMetabolismo + Clínica + Supervivencia",
          gp = gpar(fontsize = 26, fontface = "bold"), y = 0.60)
grid.text(paste("Pacientes Core:", sum(analysis$Group == "Core"),
                "  |  Divergentes:", sum(analysis$Group == "Divergent")),
          gp = gpar(fontsize = 16), y = 0.45)
grid.text(paste("Variables metabólicas analizadas:", length(metab_available),
                "  |  Significativas (p_adj<0.05):", sum(metab_res$p_adj < 0.05, na.rm = TRUE)),
          gp = gpar(fontsize = 14, col = "grey40"), y = 0.35)

# ─── Heatmap resumen de todas las variables metabólicas ──────────────────────
if (nrow(metab_res) > 0) {
  hm <- plot_heatmap_summary(analysis, metab_res, "Todas las variables metabólicas")
  if (!is.null(hm)) print(hm)
}

# ─── Sección por grupo temático metabólico ───────────────────────────────────
for (group_name in names(METAB_GROUPS)) {
  vars_in_group <- intersect(METAB_GROUPS[[group_name]], metab_available)
  if (length(vars_in_group) == 0) next

  stats_group <- metab_res %>% filter(Feature %in% vars_in_group)

  # Heatmap del grupo
  hm_g <- plot_heatmap_summary(analysis, stats_group, group_name)
  if (!is.null(hm_g)) print(hm_g)

  # Violin plots de a 6
  plots_g <- map(vars_in_group, ~plot_box_feat(.x, analysis, metab_res)) %>%
    Filter(Negate(is.null), .)

  if (length(plots_g) > 0) {
    for (i in seq(1, length(plots_g), by = 6)) {
      idx     <- i:min(i + 5, length(plots_g))
      n_cols  <- min(3, length(idx))
      n_rows  <- ceiling(length(idx) / n_cols)
      grid.arrange(
        grobs = plots_g[idx],
        ncol  = n_cols,
        nrow  = n_rows,
        top   = textGrob(paste("Metabolismo —", group_name),
                         gp = gpar(fontsize = 18, fontface = "bold"))
      )
    }
  }
}

# ─── Top 12 variables más significativas (volcán visual) ─────────────────────
if (nrow(metab_res) >= 2) {
  top12 <- head(metab_res, 12)
  top12_plots <- map(top12$Feature, ~plot_box_feat(.x, analysis, metab_res)) %>%
    Filter(Negate(is.null), .)
  if (length(top12_plots) > 0) {
    for (i in seq(1, length(top12_plots), by = 6)) {
      idx    <- i:min(i + 5, length(top12_plots))
      n_cols <- min(3, length(idx))
      n_rows <- ceiling(length(idx) / n_cols)
      grid.arrange(
        grobs = top12_plots[idx],
        ncol  = n_cols, nrow = n_rows,
        top   = textGrob("Top Variables Metabólicas Diferenciales",
                         gp = gpar(fontsize = 18, fontface = "bold"))
      )
    }
  }
}

# ─── Variables Clínicas Numéricas ───────────────────────────────────────────
if (nrow(clinical_num_res) > 0) {
  num_plots <- map(clinical_num_res$Feature, ~plot_box_feat(.x, analysis, clinical_num_res)) %>%
    Filter(Negate(is.null), .)
  if (length(num_plots) > 0) {
    for (i in seq(1, length(num_plots), by = 6)) {
      idx <- i:min(i + 5, length(num_plots))
      grid.arrange(grobs = num_plots[idx], ncol = 3,
                   top = textGrob("Variables Clínicas Numéricas",
                                  gp = gpar(fontsize = 20, fontface = "bold")))
    }
  }
}

# ─── Variables Clínicas Categóricas ─────────────────────────────────────────
clinical_to_plot <- intersect(c(
  "ajcc_pathologic_stage.diagnoses",
  "ajcc_pathologic_t.diagnoses",
  "ajcc_pathologic_n.diagnoses",
  "ajcc_pathologic_m.diagnoses",
  "tumor_grade.diagnoses",
  "morphology.diagnoses",
  "primary_diagnosis.diagnoses",
  "treatment_type.treatments.diagnoses",
  "treatment_or_therapy.treatments.diagnoses",
  "prior_treatment.diagnoses",
  "sample_type.samples",
  "tissue_type.samples",
  "tumor_descriptor.samples",
  "specimen_type.samples",
  "alcohol_history.exposures",
  "Menopausal Status",
  "Cancer Type",
  "ER", "PR", "HER2",
  "Subtype",
  "Genetic Ancestry",
  "ER_PR_HER2_Combo"
), colnames(analysis))

c_plots <- map(clinical_to_plot, ~plot_clinical(.x, analysis)) %>%
  Filter(Negate(is.null), .)

if (length(c_plots) > 0) {
  for (i in seq(1, length(c_plots), by = 2)) {
    idx <- i:min(i + 1, length(c_plots))
    grid.arrange(grobs = c_plots[idx], nrow = length(idx), ncol = 1,
                 top = textGrob("Distribución Clínica por Grupo: Core vs Divergent",
                                gp = gpar(fontsize = 20, fontface = "bold")))
  }
}

# ─── Kaplan-Meier ────────────────────────────────────────────────────────────
if (all(c("OS.time", "OS") %in% colnames(analysis))) {
  km_data <- analysis %>%
    filter(!is.na(OS.time), !is.na(OS), OS.time > 0)

  if (nrow(km_data) > 0 && length(unique(km_data$Group)) == 2) {
    fit <- survfit(Surv(OS.time, OS) ~ Group, data = km_data)
    print(ggsurvplot(fit, data = km_data,
                     pval           = TRUE,
                     pval.size      = 5,
                     conf.int       = TRUE,
                     palette        = unname(group_colors),
                     risk.table     = TRUE,
                     risk.table.cex = 1.0,
                     surv.line.size = 1.2,
                     title          = "Curva de Supervivencia: Core vs Divergent",
                     xlab           = "Tiempo (días)",
                     ylab           = "Probabilidad de Supervivencia",
                     legend.labs    = c("Core", "Divergent"),
                     theme          = base_theme +
                       theme(plot.title  = element_text(size = 22, face = "bold"),
                             axis.title  = element_text(size = 16),
                             axis.text   = element_text(size = 14),
                             legend.text = element_text(size = 14))))
  } else {
    warning("⚠️  Datos insuficientes para Kaplan-Meier tras filtrado.")
  }
} else {
  warning("⚠️  Columnas OS y/o OS.time no encontradas en el dataset metabólico.")
}

dev.off()
cat("\n✅ Reporte generado: 'Reporte_Metabolismo_Clinico_2026.pdf'\n")
cat("✅ Tabla estadística: 'Estadisticas_Metabolicas_CoreVsDivergent.csv'\n")


📊 Variables metabólicas encontradas: 40 / 40
✅ Pacientes Core: 1196  | Divergent: 20 

📈 Variables metabólicas significativas (p_adj < 0.05): 2 / 38 
✅ Tabla estadística exportada: Estadisticas_Metabolicas_CoreVsDivergent.csv


Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'Metabolismo — Capacidad de Uso y Eficiencia de Activación' in 'mbcsToSbcs': - substituted for — (U+2014)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'Metabolismo — Índice de Warburg' in 'mbcsToSbcs': - substituted for — (U+2014)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'Metabolismo — Vías Metabólicas (SA)' in 'mbcsToSbcs': - substituted for — (U+2014)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'Metabolismo — Vías Metabólicas (SA)' in 'mbcsToSbcs': - substituted for — (U+2014)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'Metabolismo — Redox, Flujo y Anabolismo' in 'mbcsToSbcs': - substituted for — (U+2014)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'Metabolismo 

pdf 
  2


✅ Reporte generado: 'Reporte_Metabolismo_Clinico_2026.pdf'
✅ Tabla estadística: 'Estadisticas_Metabolicas_CoreVsDivergent.csv'


In [8]:
install.packages("pheatmap")

Installing package into ‘/Users/eduardoruiz/Library/R/arm64/4.4/library’
(as ‘lib’ is unspecified)




The downloaded binary packages are in
	/var/folders/x0/j78w4f8n0_z4l575_k16n2k40000gn/T//RtmpPiHpQZ/downloaded_packages
